# 3. Web Scraping

Cuando un sitio no ofrece API, los datos pueden obtenerse con **web scraping**: descargar el HTML con `requests` y extraer la información con `BeautifulSoup`.

Para practicar la técnica se usa **https://books.toscrape.com**, un sitio creado específicamente para aprender scraping (permite el acceso automatizado, sin restricciones de `robots.txt`). El flujo es el mismo que se aplicaría a cualquier sitio real:

1. Descargar la página (`requests.get`).
2. Parsear el HTML (`BeautifulSoup`).
3. Localizar los elementos con selectores (etiquetas, clases CSS).
4. Estructurar los datos en un `DataFrame` y guardarlos.

> **Buenas prácticas:** revisar siempre `robots.txt` y los términos del sitio, espaciar las peticiones y preferir una API oficial cuando exista (como se hizo con Yelp en el notebook 2).

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

URL_BASE = "https://books.toscrape.com/catalogue/page-{}.html"

# Paso 1 y 2: descargar y parsear la primera página
respuesta = requests.get(URL_BASE.format(1), timeout=15)
respuesta.raise_for_status()
respuesta.encoding = "utf-8"  # el sitio no declara charset y requests asume latin-1
sopa = BeautifulSoup(respuesta.text, "html.parser")

print("Status:", respuesta.status_code)
print("Título de la página:", sopa.title.get_text(strip=True))
print("Libros en la página:", len(sopa.select("article.product_pod")))

Status: 200
Título de la página: All products | Books to Scrape - Sandbox
Libros en la página: 20


## 3.1 Extracción de varias páginas

Cada libro está en un `<article class="product_pod">`: el título en el atributo `title` del enlace, el precio en `<p class="price_color">` y la calificación como clase CSS de `<p class="star-rating">`.

In [2]:
RATING_TEXTO = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
libros = []

for pagina in range(1, 6):  # primeras 5 páginas (20 libros c/u)
    r = requests.get(URL_BASE.format(pagina), timeout=15)
    r.raise_for_status()
    r.encoding = "utf-8"
    sopa = BeautifulSoup(r.text, "html.parser")
    for libro in sopa.select("article.product_pod"):
        libros.append({
            "titulo": libro.h3.a["title"],
            "precio": libro.select_one("p.price_color").get_text(strip=True),
            "rating": RATING_TEXTO[libro.select_one("p.star-rating")["class"][1]],
            "disponible": "In stock" in libro.select_one("p.instock.availability").get_text(),
        })
    time.sleep(0.5)  # pausa entre peticiones: no sobrecargar el servidor

df_libros = pd.DataFrame(libros)
print("Libros extraídos:", len(df_libros))
df_libros.head()

Libros extraídos: 100


,titulo,precio,rating,disponible
0,A Light in the Attic,£51.77,3,True
1,Tipping the Velvet,£53.74,1,True
2,Soumission,£50.10,1,True
3,Sharp Objects,£47.82,4,True
4,Sapiens: A Brief History of Humankind,£54.23,5,True


## 3.2 Limpieza y análisis de lo extraído

El precio llega como texto (`£51.77`): se extrae la parte numérica con una expresión regular — el mismo tipo de limpieza aplicado a los teléfonos en `1_EDA.ipynb`.

In [3]:
df_libros["precio_gbp"] = df_libros["precio"].str.extract(r"(\d+\.\d+)")[0].astype(float)

print("=== Estadísticas de precio (£) ===")
print(df_libros["precio_gbp"].describe().round(2))

print("\n=== Distribución de calificaciones ===")
t = df_libros["rating"].value_counts().sort_index().to_frame("libros")
t["%"] = (t["libros"] / len(df_libros) * 100).round(1)
print(t)

print("\n=== Precio medio por calificación ===")
print(df_libros.groupby("rating")["precio_gbp"].agg(["mean", "count"]).round(2))

=== Estadísticas de precio (£) ===
count    100.00
mean      34.56
std       14.64
min       10.16
25%       19.90
50%       34.78
75%       47.97
max       58.11
Name: precio_gbp, dtype: float64

=== Distribución de calificaciones ===
        libros     %
rating              
1           22  22.0
2           19  19.0
3           22  22.0
4           18  18.0
5           19  19.0

=== Precio medio por calificación ===
         mean  count
rating              
1       35.52     22
2       35.91     19
3       36.84     22
4       33.98     18
5       30.01     19


In [4]:
# Guardar el resultado del scraping
df_libros.to_csv("../../Datasets/libros_scraping.csv", index=False, encoding="utf-8")
print("Guardado: Datasets/libros_scraping.csv")

Guardado: Datasets/libros_scraping.csv


## Conclusiones

- Se extrajeron **100 libros** (5 páginas) con título, precio, calificación y disponibilidad, usando `requests` + `BeautifulSoup`.
- El precio medio ronda las **£35** y la calificación no muestra relación con el precio — en este sitio de práctica los datos son generados al azar.
- La técnica es directamente trasladable al dominio del proyecto: por ejemplo, scrapear cartas y precios de los sitios web de los restaurantes de Miami que no publican esa información en la API de Yelp.
- Diferencia clave con el notebook 2: la **API** entrega datos estructurados y estables (JSON), mientras que el **scraping** depende de la estructura HTML del sitio y puede romperse si el sitio cambia — por eso la API es siempre la primera opción.